In [ ]:
!pip install dotenv

In [ ]:
!pip install langchain pymupdf faiss-cpu google-generativeai langchain-community

In [1]:
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings.base import Embeddings
from typing import List
import google.generativeai as genai
import asyncio

d:\Ky 4\vietnamese-speech-chatbox\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
import os

secret_key = os.getenv("SECRET_KEY")

genai.configure(api_key=secret_key)

In [3]:
models = genai.list_models()
for model in models:
    print(model.name)

models/embedding-gecko-001
models/gemini-1.0-pro-vision-latest
models/gemini-pro-vision
models/gemini-1.5-pro-latest
models/gemini-1.5-pro-001
models/gemini-1.5-pro-002
models/gemini-1.5-pro
models/gemini-1.5-flash-latest
models/gemini-1.5-flash-001
models/gemini-1.5-flash-001-tuning
models/gemini-1.5-flash
models/gemini-1.5-flash-002
models/gemini-1.5-flash-8b
models/gemini-1.5-flash-8b-001
models/gemini-1.5-flash-8b-latest
models/gemini-1.5-flash-8b-exp-0827
models/gemini-1.5-flash-8b-exp-0924
models/gemini-2.5-pro-exp-03-25
models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-04-17
models/gemini-2.5-flash-preview-05-20
models/gemini-2.5-flash-preview-04-17-thinking
models/gemini-2.5-pro-preview-05-06
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-preview-image-generation
models/gemini-2.0-flash-lite-preview

In [4]:
pdf_path = r"C:\Users\5530\Downloads\att.pdf"
loader = PyMuPDFLoader(pdf_path)
documents = loader.load()

In [12]:
text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
)
chunks = text_splitter.split_documents(documents)

for chunk in chunks:
    print(chunk.page_content)
    break

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Exper

In [13]:
class GeminiEmbeddings(Embeddings):
    def __init__(self, model_name="models/text-embedding-004"):
        self.model_name = model_name

    def embed_query(self, query: str) -> List[float]:
        try:
            result = genai.embed_content(
                model=self.model_name,
                content=query
            )
            return result['embedding']
        except Exception as e:
            print(f"Error embedding query: {e}")
            return []

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        embeddings = []
        for text in texts:
            try:
                result = genai.embed_content(
                    model=self.model_name,
                    content=text
                )
                embeddings.append(result['embedding'])
            except Exception as e:
                print(f"Error embedding document: {e}")
                embeddings.append([])
        return embeddings

embeddings = GeminiEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)

In [15]:
class SmartChatbot:
    def __init__(self, vectorstore, model_name="models/gemini-2.0-flash-exp"):
        self.vectorstore = vectorstore
        self.model_name = model_name

    async def answer_question(self, question: str) -> str:
        try:
            # Retrieve relevant documents using the vectorstore
            related_docs = self.vectorstore.similarity_search(question, k=3)
            context = "\n\n".join([doc.page_content for doc in related_docs])

            # Generate a response using the LLM with context
            prompt = (
                f"You are a helpful assistant. Use the following context to answer:\n\n"
                f"{context}\n\n"
                f"Question: {question}\n"
                f"Answer:"
            )

            # Use GenerativeModel from google.generativeai
            model = genai.GenerativeModel(self.model_name)
            response = model.generate_content(prompt)
            return response.text
        except Exception as e:
            return f"Error: {e}"


async def simulate_typing(text):
    for char in text:
        print(char, end='', flush=True)
        await asyncio.sleep(0.0002)
    print()

async def main():
    # Initialize the chatbot
    chatbot = SmartChatbot(vectorstore)

    # Ask a question
    question = "What is the main topic of the document? Can you describe it in detail?"

    print(f"\nYou: {question}")
    print("\nBot is typing...", end='\n')
    await asyncio.sleep(1)
    response = await chatbot.answer_question(question)
    print(" " * 15, end='\r')  # Clear "Bot is typing..."
    print()
    await simulate_typing(response)

await main()


You: What is the main topic of the document? Can you describe it in detail?

Bot is typing...
               
The document discusses attention visualizations in the context of a language model. Specifically, it examines how attention mechanisms in the encoder self-attention layers of a model capture long-distance dependencies, perform anaphora resolution, and learn to reflect the structure of sentences.

Figure 3 illustrates how different attention heads attend to distant dependencies, exemplified by the phrase "making...more difficult". Figure 4 shows attention heads involved in anaphora resolution, focusing on the word "its". Figure 5 presents examples of attention heads exhibiting behavior related to the structure of sentences, suggesting that different heads learn to perform different tasks. The document provides these visualizations to understand how the attention mechanism works within the model.



# https://ai.google.dev/gemini-api/docs/rate-limits

# https://ai.google.dev/gemini-api/docs/pricing